In [74]:
from clients.vecdb_client import VectorDBClient
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv

In [75]:
load_dotenv()

True

In [76]:
with open('arxiv_data_cache.json', 'r', encoding='utf-8') as file:
    points = json.load(file)

In [77]:
qd_client = VectorDBClient()
qd_client.delete_collection()
qd_client.make_collection()
qd_client.upsert_points(points)

Collection arxiv-rag deleted!
Collection arxiv-rag created!


In [78]:
# ground_truth = pd.read_csv('./generate_data/ground-truth-data.csv', dtype={'entry_id': str})

# from clients.vecdb_client import VectorDBClient
# import pandas as pd
# import numpy as np
# import json

# ground_truth = pd.read_csv('./generate_data/ground-truth-data.csv', dtype={'entry_id': str})

# def hit_rate(relevance_matrix: np.ndarray) -> float:
#     return np.any(relevance_matrix, axis = 1).mean()

# def mrr(relevance_matrix: np.ndarray) -> float:
#     hits = np.any(relevance_matrix, axis=1)
#     ranks = np.argmax(relevance_matrix, axis=1).astype(float) + 1
#     return np.sum((1 / ranks) * hits) / len(relevance_matrix)

# def ndcg_at_k(relevance_matrix: np.ndarray) -> float:
#     hits = np.any(relevance_matrix, axis=1)
#     ranks = np.argmax(relevance_matrix, axis=1) + 1
#     dcg = np.where(hits, 1 / np.log2(ranks + 1), 0)
#     idcg = 1 / np.log2(2)
#     return (dcg / idcg).mean()

# def map_at_k(relevance_matrix: np.ndarray) -> float:
#     hits = np.any(relevance_matrix, axis=1)
#     ranks = np.argmax(relevance_matrix, axis=1) + 1
#     map = np.where(hits, 1 / ranks, 0).mean()
#     return map

# def run_evaluation(ground_truth: pd.DataFrame, search_function: VectorDBClient = qd_client, top_k:int = 10):
#     ground_truth_dict = ground_truth.to_dict(orient='records')
#     limit = top_k
#     bool_match_list = []
#     for entry in ground_truth_dict:
#         entry_question = entry['question']
#         entry_id = entry['entry_id']
#         search = search_function.search(entry_question, limit = limit)
#         matches = [result.payload.get('entry_id') == entry_id for result in search.points]
#         matches += [False] * (limit - len(matches))
#         bool_match_list.append(matches)

#     arr = np.array(bool_match_list)
#     return {
#         f'hit_rate_k_{top_k}': hit_rate(arr),
#         f'mrr_k_{top_k}': mrr(arr),
#         f'ndcg_k_{top_k}': ndcg_at_k(arr),
#         f'map_k_{top_k}': map_at_k(arr)
#     }

# run_evaluation(ground_truth)

In [79]:
from clients.llm_client import LLMClient

In [80]:
llm_client = LLMClient(os.environ["GEMINI_API_KEY"])

In [81]:
data = pd.read_csv('./generate_data/response-evaluation-subset.csv')

In [82]:
def build_prompt(query_results, query) -> str:
    system_prompt = (
        "You are a chatbot designed to synthesize and summarize the results of a search "
        "on the Arxiv database of papers in the last 30 days based on the user's queries. "
        f"QUERY: {query}\n\n"
        "Based on the query, the search returned these documents that might be relevant. Review the following retrieved documents carefully:\n\n"
    )

    formatted_results = []
    iter = len(query_results.points)
    for i in range(0, iter):
        # Dynamically handle an arbitrary number of authors using .join()
        authors_list = query_results.points[i].payload.get("authors", ["Unknown Author"])
        authors_str = ", ".join(authors_list)
        categories_list = query_results.points[i].payload.get("categories", ["Unknown category"])
        categories_str = ", ".join(categories_list)
        title = query_results.points[i].payload.get("title", "Untitled")

        # Properly utilize f-strings for variable interpolation
        result_block = (
            f"RESULT {i+1}:\n"
            f"    Title: {title}\n"
            f"    Authors: {authors_str}\n"
            f"    Categories: {categories_str}\n"
            f"    Abstract: {query_results.points[i].payload.get("summary", ["Unknown abstract"])}\n"
            f"    Published date: {query_results.points[i].payload.get("published", ["Unknown date"])}\n"
            f"    Entry ID: {query_results.points[i].payload.get("entry_id", ["Unknown ID"])}\n"
        )
        formatted_results.append(result_block)

    # Assemble the final prompt efficiently
    final_prompt = system_prompt + "\n".join(formatted_results)
    final_prompt_add = final_prompt + "\nNow, answer the user's query."
    return final_prompt_add.strip()

In [83]:
def generate_responses():
    results = []
    for item in data.to_dict(orient='records'):
        query = item['question']
        retrieval = qd_client.search(query)
        prompt = build_prompt(retrieval, query)
        response = llm_client.prompt_llm(prompt)

        payloads = [point.payload for point in retrieval.points]
        
        results.append({
            **item,
            'retrieved_context': payloads,
            'response': response.choices[0].message.content
        })
        break
        
    return results

In [84]:
results_list = generate_responses()

In [85]:
print(results_list)

[{'question': 'What method uses a system of specialized software entities to mimic the collaborative review process of a medical department?', 'entry_id': 'http://arxiv.org/abs/2604.16175v1', 'generated_at': '2026-04-23 04:43:51.926898+00:00', 'retrieved_context': [{'title': 'MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation', 'categories': ['cs.AI', 'cs.CV'], 'authors': ['Yi Lin', 'Yihao Ding', 'Yonghui Wu', 'Yifan Peng'], 'summary': 'Automated 3D radiology report generation often suffers from clinical hallucinations and a lack of the iterative verification found in human practice. While recent Vision-Language Models (VLMs) have advanced the field, they typically operate as monolithic "black-box" systems without the collaborative oversight characteristic of clinical workflows. To address these challenges, we propose MARCH (Multi-Agent Radiology Clinical Hierarchy), a multi-agent framework that emulates the professional hierarchy of radiology departments and assi

In [86]:
with open('evaluation_data.json', 'w', encoding='utf-8') as f:
    json.dump(results_list, f, indent=4)

In [87]:
with open('evaluation_data.json', 'r', encoding='utf-8') as f:
    raw_response = json.load(f)

In [88]:
print(raw_response)

[{'question': 'What method uses a system of specialized software entities to mimic the collaborative review process of a medical department?', 'entry_id': 'http://arxiv.org/abs/2604.16175v1', 'generated_at': '2026-04-23 04:43:51.926898+00:00', 'retrieved_context': [{'title': 'MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation', 'categories': ['cs.AI', 'cs.CV'], 'authors': ['Yi Lin', 'Yihao Ding', 'Yonghui Wu', 'Yifan Peng'], 'summary': 'Automated 3D radiology report generation often suffers from clinical hallucinations and a lack of the iterative verification found in human practice. While recent Vision-Language Models (VLMs) have advanced the field, they typically operate as monolithic "black-box" systems without the collaborative oversight characteristic of clinical workflows. To address these challenges, we propose MARCH (Multi-Agent Radiology Clinical Hierarchy), a multi-agent framework that emulates the professional hierarchy of radiology departments and assi

In [89]:
def build_faithfulness_prompt(entry: dict) -> str:
    """
    Constructs an evaluation prompt to measure how faithfully the generated 
    response adheres to the retrieved contexts.
    """
    retrieved_contexts = entry.get("retrieved_context", [])
    
    # Extract and format the contexts to maximize legibility for the judging LLM
    formatted_contexts = []
    for i, ctx in enumerate(retrieved_contexts):
        title = ctx.get("title", "No Title")
        summary = ctx.get("summary", "No Summary")
        formatted_contexts.append(f"--- Context {i+1} ---\nTitle: {title}\nSummary: {summary}")
    
    contexts_text = "\n\n".join(formatted_contexts)
    response = entry.get("response", "")

    prompt = f"""You are an objective evaluator measuring the 'faithfulness' of a generated response.
Faithfulness evaluates whether the response is strictly grounded in the provided context on a scale of 1 to 3:
- 1 (Low): The response contains information entirely unsupported by, or contradictory to, the retrieved context.
- 2 (Medium): The response is partially supported but includes some hallucinations or unverified external claims.
- 3 (High): The response is fully supported by the retrieved context without introducing external information.

### Retrieved Contexts:
{contexts_text}

### Generated Response:
{response}

Please analyze the content and context of the generated answer in relation to the retrieved context and provide your evaluation in parsable JSON without using code blocks:
{{ "explanation_faithfulness": "[Provide a brief explanation for your evaluation]", "faithfulness": 1 | 2 | 3}}""".strip()
    
    return prompt


def build_relevance_prompt(entry: dict) -> str:
    """
    Constructs an evaluation prompt to measure how relevant the generated 
    response is to the original user question.
    """
    question = entry.get("question", "")
    response = entry.get("response", "")

    prompt = f"""You are an objective evaluator measuring the 'response relevance' of a generated response.
Response relevance evaluates how well the generated response directly answers the provided question on a scale of 1 to 3:
- 1 (Low): The response is completely irrelevant to the core intent of the question.
- 2 (Medium): The response partially addresses the question but misses the core intent or includes significant tangential information.
- 3 (High): The response directly, clearly, and comprehensively answers the question.

### Question:
{question}

### Generated Response:
{response}

Please analyze the content and context of the generated answer in relation to the question and provide your evaluation in parsable JSON without using code blocks:
{{ "explanation_relevance": "[Provide a brief explanation for your evaluation]", "relevance": 1 | 2 | 3}}""".strip()
    
    return prompt

In [90]:
print(build_faithfulness_prompt(raw_response[0]))

You are an objective evaluator measuring the 'faithfulness' of a generated response.
Faithfulness evaluates whether the response is strictly grounded in the provided context on a scale of 1 to 3:
- 1 (Low): The response contains information entirely unsupported by, or contradictory to, the retrieved context.
- 2 (Medium): The response is partially supported but includes some hallucinations or unverified external claims.
- 3 (High): The response is fully supported by the retrieved context without introducing external information.

### Retrieved Contexts:
--- Context 1 ---
Title: MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation
Summary: Automated 3D radiology report generation often suffers from clinical hallucinations and a lack of the iterative verification found in human practice. While recent Vision-Language Models (VLMs) have advanced the field, they typically operate as monolithic "black-box" systems without the collaborative oversight characteristic of clin

In [91]:
print(build_relevance_prompt(raw_response[0]))

You are an objective evaluator measuring the 'response relevance' of a generated response.
Response relevance evaluates how well the generated response directly answers the provided question on a scale of 1 to 3:
- 1 (Low): The response is completely irrelevant to the core intent of the question.
- 2 (Medium): The response partially addresses the question but misses the core intent or includes significant tangential information.
- 3 (High): The response directly, clearly, and comprehensively answers the question.

### Question:
What method uses a system of specialized software entities to mimic the collaborative review process of a medical department?

### Generated Response:
The method that uses a system of specialized software entities to mimic the collaborative review process of a medical department is called **MARCH (Multi-Agent Radiology Clinical Hierarchy)**.

As detailed in the paper *MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation* (Result 1), this fram

In [92]:
with open('evaluation_data.json', 'r', encoding='utf-8') as f:
    raw_response = json.load(f)

In [97]:
def run_judges(eval_data: list[dict]):
    import json
    eval_list = []
    for item in eval_data:
        faithfulness_prompt = build_faithfulness_prompt(item)
        relevance_prompt = build_relevance_prompt(item)
        
        faithfulness_eval = llm_client.prompt_llm(faithfulness_prompt)
        relevance_eval = llm_client.prompt_llm(relevance_prompt)
        
        f_content = faithfulness_eval.choices[0].message.content
        r_content = relevance_eval.choices[0].message.content
        print(item)
        print(f_content)
        print(r_content)
        
        # Parse JSON strings into dictionaries, with error handling for LLM output variances
        try:
            f_dict = json.loads(f_content)
            r_dict = json.loads(r_content)
            
            combined_eval = {
                **item,
                **f_dict,
                **r_dict,                
            }
            eval_list.append(combined_eval) # Appends the evaluated data to the list
            
        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            # Consider appending a fallback dictionary here if parsing fails
            
    return eval_list

In [98]:
final = run_judges(raw_response)

{'question': 'What method uses a system of specialized software entities to mimic the collaborative review process of a medical department?', 'entry_id': 'http://arxiv.org/abs/2604.16175v1', 'generated_at': '2026-04-23 04:43:51.926898+00:00', 'retrieved_context': [{'title': 'MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation', 'categories': ['cs.AI', 'cs.CV'], 'authors': ['Yi Lin', 'Yihao Ding', 'Yonghui Wu', 'Yifan Peng'], 'summary': 'Automated 3D radiology report generation often suffers from clinical hallucinations and a lack of the iterative verification found in human practice. While recent Vision-Language Models (VLMs) have advanced the field, they typically operate as monolithic "black-box" systems without the collaborative oversight characteristic of clinical workflows. To address these challenges, we propose MARCH (Multi-Agent Radiology Clinical Hierarchy), a multi-agent framework that emulates the professional hierarchy of radiology departments and assig

In [99]:
print(final)

[{'question': 'What method uses a system of specialized software entities to mimic the collaborative review process of a medical department?', 'entry_id': 'http://arxiv.org/abs/2604.16175v1', 'generated_at': '2026-04-23 04:43:51.926898+00:00', 'retrieved_context': [{'title': 'MARCH: Multi-Agent Radiology Clinical Hierarchy for CT Report Generation', 'categories': ['cs.AI', 'cs.CV'], 'authors': ['Yi Lin', 'Yihao Ding', 'Yonghui Wu', 'Yifan Peng'], 'summary': 'Automated 3D radiology report generation often suffers from clinical hallucinations and a lack of the iterative verification found in human practice. While recent Vision-Language Models (VLMs) have advanced the field, they typically operate as monolithic "black-box" systems without the collaborative oversight characteristic of clinical workflows. To address these challenges, we propose MARCH (Multi-Agent Radiology Clinical Hierarchy), a multi-agent framework that emulates the professional hierarchy of radiology departments and assi